# LifeMate: Gipformer + Qwen on Kaggle

Import this notebook into Kaggle, attach your **private** `amnpham/zalochallenge` dataset, select a GPU, and enable Internet for setup (GitHub, packages, model downloads). No Kaggle token is needed inside the notebook.

Run cells in order. The default uses **25 fixed train development IDs with audio**. Set `IDS_FILE=None` and select `LIMIT` to run a different range; previously inspected IDs are not holdout. To generate public-test predictions, select `public_test` and disable train evaluation. Test labels must never be generated or used for tuning.

Inference uses local models with Hugging Face offline flags. This does not simulate a network-disabled Docker container. The competition limit remains **30 seconds/sample**; the public notebook budget is 27 seconds; this train notebook uses 90 seconds for diagnosis. A larger T4 development budget is not evidence of A30 compliance.

Direct behavior instructions are in English; no demonstrations are injected. Vietnamese transcripts and user/tool data stay in their original language. The language change is not a measured accuracy or speed improvement.


Pipeline: **controlled train experiments**. Default: single agent native actions with strict output parsing, unchanged arguments and execution status. Old experiments have been removed. Memory never supplies evidence. The 90-second budget is for correctness debugging, not competition compliance.


Direct baseline: run the single pipeline on a fixed train development ID list. Compare the frozen base checkpoint and a trained adapter; do not mix results from different prompts or transcript sources. Keep all other settings identical. Start with `TRAIN_TRANSCRIPTS=True` to isolate tool use, then repeat with ASR. Run configurations sequentially on the GPU; each run uses a fresh ToolEnv per sample. Previously inspected IDs are development data, not holdout. No public-test tuning.


In [ ]:
# Edit these settings before running.
REPO_URL = "https://github.com/AmnO-O/ZolaAIC.git"
REVISION = "main"  # Use a commit hash to reproduce a specific version.
REPO_DIR = "/kaggle/working/ZolaAIC"

DATASET_ROOT = "/kaggle/input/datasets/amnpham/zalochallenge/train+public_test"
SPLIT = "train"  # "train", "public_test", or "private_test"
IDS_FILE = "configs/ids_25.json"  # "configs/ids_25.json", or None
LIMIT = None        # Set None when using IDS_FILE; or e.g. 1 / 25 / 300
BUDGET_SECONDS = 90  # DEVELOPMENT ONLY: prioritize correctness; not competition compliant
MAX_TURNS = 16  # Bounds decisions, including protocol corrections
PROMPT_CACHE = True  # Reuse exact prompt prefixes within each query
DETERMINISTIC = True
TRAIN_TRANSCRIPTS = True  # Set False for REAL ASR; True bypasses ASR using train ground-truth
EVALUATE_TRAIN = True        # Automatically skipped for test splits
REFERENCE_REPLAY = False    # Optional separate TRAIN evaluator sessions only
RUN_UNIT_TESTS = True
STEP_LOGGING = True          # Save full inputs/outputs for every sample
LIVE_STEPS = True            # Print full step inputs/outputs while running

# Optional existing local model directories. Leave blank to download pinned assets.
LLM_PATH = ""
ASR_PATH = ""
ADAPTER_PATH = ""  # Optional LOCAL adapter; install requirements-train.txt to load it


## 1. Clone the source and install dependencies

This installs the repository's pinned direct dependencies, including PyTorch. If Kaggle asks for a kernel restart after installation, restart before continuing. The notebook does not overwrite an existing checkout; use a fresh session or a different `REPO_DIR` to select another revision.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

repo = Path(REPO_DIR).resolve()
if repo.exists():
    actual_root = Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=repo, text=True).strip()).resolve()
    if actual_root != repo:
        raise RuntimeError("REPO_DIR must point to the repository root")
    remote = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Unexpected repository at {repo}; choose a new REPO_DIR")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo, text=True).strip()
    if dirty:
        raise RuntimeError("Checkout has local edits. Preserve them and choose a new REPO_DIR.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)

# Works for a remote branch, tag, or exact commit; never merges local changes.
subprocess.run(["git", "fetch", "origin", REVISION], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True)
if not (repo / "src/lifemate/prompt_assets/system.txt").is_file():
    raise RuntimeError("This revision predates the clean pipeline; select its notebook or a newer revision.")

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Source commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)


## 2. Check the GPU and dataset path

Only the selected split's catalog is checked here. No environment directory is inspected. If detection fails, set `DATASET_ROOT` to the parent of `train/` and `public_test/` shown by Kaggle's Input panel.


In [ ]:
import torch

if SPLIT not in {"train", "public_test", "private_test"}:
    raise ValueError("Unknown split")
if LIMIT is not None and (isinstance(LIMIT, bool) or not isinstance(LIMIT, int) or LIMIT < 1):
    raise ValueError("LIMIT must be a positive integer or None")
if IDS_FILE:
    ids_path = repo / IDS_FILE
    if not ids_path.is_file():
        raise FileNotFoundError(f"Missing IDs file: {ids_path}")
    print("Using specific sample IDs from:", ids_path)
import math
if not math.isfinite(BUDGET_SECONDS) or BUDGET_SECONDS <= 0:
    raise ValueError("BUDGET_SECONDS must be positive and finite")
if TRAIN_TRANSCRIPTS and SPLIT != "train":
    raise ValueError("Reference transcripts are train-only")
if REFERENCE_REPLAY and (SPLIT != "train" or not EVALUATE_TRAIN):
    raise ValueError("Reference replay requires train evaluation")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle notebook settings")
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

candidates = [Path(DATASET_ROOT) / SPLIT]
if DATASET_ROOT == "/kaggle/input/datasets/amnpham/zalochallenge/train+public_test":
    candidates.append(Path("/kaggle/input/zalochallenge/train+public_test") / SPLIT)
split_dir = next((p for p in candidates if (p / f"{SPLIT}_tools_catalog.json").is_file()), None)
if split_dir is None:
    raise FileNotFoundError("Split catalog not found. Correct DATASET_ROOT using the Kaggle Input panel.")
print("Selected split:", split_dir)
if BUDGET_SECONDS > 30:
    print(f"Note: Development budget ({BUDGET_SECONDS}s) exceeds competition 30s limit for baseline alignment.")

if not isinstance(MAX_TURNS, int) or MAX_TURNS < 1:
    raise ValueError("MAX_TURNS must be positive")


## 3. Prepare local models and run source tests

Weights download only in this setup cell. Models use the revisions recorded in `configs/baseline.json`. Steerability source is already included in the repository. The tests use synthetic fixtures and do not load the LLM.


In [ ]:
for component, override in (("llm", LLM_PATH), ("asr", ASR_PATH)):
    if override:
        model_path = Path(override).resolve()
    else:
        subprocess.run([sys.executable, "scripts/download_models.py", "--component", component], cwd=repo, check=True)
        model_path = repo / "models" / component
    if not model_path.is_dir():
        raise FileNotFoundError(f"Missing {component} model directory: {model_path}")
    os.environ[f"LIFEMATE_{component.upper()}_PATH"] = str(model_path)

os.environ.update(
    HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONHASHSEED="42",
    CUBLAS_WORKSPACE_CONFIG=":4096:8", LIFEMATE_ASR_KIND="gipformer",
    LIFEMATE_ASR_PRECISION="int8", LIFEMATE_ASR_THREADS="4",
    LIFEMATE_PROMPT_CACHE="1" if PROMPT_CACHE else "0",
    LIFEMATE_DETERMINISTIC="1" if DETERMINISTIC else "0",
)
if RUN_UNIT_TESTS:
    subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=repo, env=dict(os.environ), check=True)
print("Local assets ready; inference download paths are disabled.")

if ADAPTER_PATH:
    os.environ["LIFEMATE_ADAPTER_PATH"] = ADAPTER_PATH
else:
    os.environ.pop("LIFEMATE_ADAPTER_PATH", None)


## 4. Run inference

Each run writes to a new directory under `/kaggle/working/runs/`. The process loads models once, then uses exactly one ToolEnv per prediction sample. All calls made on that session are exported. `submission.trace.jsonl` contains timing and failure diagnostics. Native ASR calls check deadlines before and after a blocking decode.

A train-transcript run measures the tool agent without ASR; it must not be presented as audio performance. A test run produces predictions only. The script records sample failures in the trace, so review the summary even when the process exits successfully.




Controller fixes preserve explicit amounts, recipients, dates, addresses and filters. Known historical tool results can ground IDs/phones without replay. Pruning defaults off; if enabled, the controller retains the full catalog for recovery. If a session cannot be created/exported, later samples still run, but no fake state hash is generated: the row is omitted, failure is recorded, and the process exits nonzero after processing the batch. Partial output is not a complete submission.


In [ ]:
if BUDGET_SECONDS >= 30:
    print("DEVELOPMENT BUDGET: results do not establish the 30-second competition requirement.")
from datetime import datetime, timezone
import uuid
import time

run_dir = Path("/kaggle/working/runs") / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
run_dir.mkdir(parents=True)
submission = run_dir / "submission.jsonl"
command = [sys.executable, "scripts/infer.py", "--split-dir", str(split_dir),
           "--backend", "lifemate.backend:create_backend", "--budget-seconds", str(BUDGET_SECONDS),
           "--output", str(submission)]
if IDS_FILE:
    command += ["--ids", str(repo / IDS_FILE)]
elif LIMIT is not None:
    command += ["--limit", str(LIMIT)]
if TRAIN_TRANSCRIPTS:
    command.append("--train-transcripts")
if STEP_LOGGING:
    command += ["--step-log", str(run_dir / "steps.jsonl")]
    if LIVE_STEPS:
        command.append("--verbose-steps")

def run_logged(command, log_name):
    with (run_dir / log_name).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return process.wait()

command += ["--max-turns", str(MAX_TURNS)]
started = time.monotonic()
returncode = run_logged(command, "inference.log")
run_report = {
    "policy": "single", "budget_seconds": BUDGET_SECONDS, "max_turns": MAX_TURNS, "adapter_path": ADAPTER_PATH,
    "commit": COMMIT, "split": SPLIT, "split_dir": str(split_dir), "limit": LIMIT,
    "train_transcripts": TRAIN_TRANSCRIPTS, "inference_exit_code": returncode,
    "step_logging": STEP_LOGGING, "prompt_cache": PROMPT_CACHE,
    "requested_deterministic": DETERMINISTIC,
    "process_seconds_including_model_load": time.monotonic() - started,
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "model_config": json.loads((repo / "configs/baseline.json").read_text()),
    "model_paths": {key: os.environ[key] for key in ("LIFEMATE_LLM_PATH", "LIFEMATE_ASR_PATH")},
}
(run_dir / "run_report.json").write_text(json.dumps(run_report, indent=2), encoding="utf-8")
print("Output directory:", run_dir)
if returncode:
    raise RuntimeError(f"Inference failed with exit code {returncode}; see inference.log")


## 5. Optional train-only diagnostic score

This evaluator reads train labels after prediction in a separate process. It never feeds labels to the agent. The F1 calculation is diagnostic, not a verified official competition score. `REFERENCE_REPLAY=True` additionally creates separate gold evaluator sessions and must never be used in submitted inference; it is disabled by default.


In [ ]:
if SPLIT == "train" and EVALUATE_TRAIN:
    summary_command = [sys.executable, "scripts/evaluate_run.py", str(run_dir),
                       "--train-dir", str(split_dir)]
    run_logged(summary_command, "evaluation_summary.log")

    evaluation_command = [sys.executable, "scripts/evaluate.py", "--predictions", str(submission),
                          "--train-dir", str(split_dir), "--output", str(run_dir / "evaluation.json")]
    if REFERENCE_REPLAY:
        evaluation_command.append("--reference-replay")
    evaluation_returncode = run_logged(evaluation_command, "evaluation.log")
    run_report["evaluation_exit_code"] = evaluation_returncode
    run_report["reference_replay"] = REFERENCE_REPLAY
    (run_dir / "run_report.json").write_text(json.dumps(run_report, indent=2), encoding="utf-8")
    if evaluation_returncode:
        raise RuntimeError("Train evaluation failed; see evaluation.log")
else:
    print("Scoring skipped. Test splits are prediction-only.")


## 6. Inspect and download results

Download `submission.jsonl` from the output directory in Kaggle's file browser, or save a notebook version to retain outputs. Keep notebooks and outputs containing restricted data private. Review trace errors and deadline counts before interpreting any score.


In [ ]:
from collections import Counter
traces = [json.loads(line) for line in submission.with_suffix(".trace.jsonl").read_text().splitlines() if line.strip()]
print("Prediction rows:", sum(1 for line in submission.open() if line.strip()))
print("Trace rows:", len(traces))
print("Trace fields:", sorted(traces[0]) if traces else [])
print("Termination reasons:", dict(Counter(row.get("termination", "not_recorded") for row in traces)))
times = [row["elapsed_seconds"] for row in traces if "elapsed_seconds" in row]
if times:
    print("Mean / max sample seconds:", round(sum(times) / len(times), 3), round(max(times), 3))
    print("Samples taking >=30 seconds:", sum(value >= 30 for value in times))
    print("Sample timing excludes process model loading; see run_report.json for process time.")
print("Download:", submission)
print("Run metadata:", run_dir / "run_report.json")
if (run_dir / "evaluation.json").is_file():
    metrics = json.loads((run_dir / "evaluation.json").read_text())
    print(json.dumps({key: value for key, value in metrics.items() if key != "results"}, indent=2))


Every query logs decision inputs/outputs, validation, actual tool calls/results, errors and termination. Model stop is not verified task success.


In [ ]:
QUERY_ID = ""  # e.g. train_0002; blank shows the first query

from html import escape
from IPython.display import HTML, display

step_file = run_dir / "steps.jsonl"
if not step_file.is_file():
    print("No detailed log. Enable STEP_LOGGING before running inference.")
else:
    selected_id = QUERY_ID or None
    sample_ids = []
    displayed = 0
    with step_file.open(encoding="utf-8") as source:
        for line in source:
            event = json.loads(line)
            if event["stage"] == "sample.input":
                sample_ids.append(event["id"])
            if selected_id is None:
                selected_id = event["id"]
            if event["id"] != selected_id:
                continue
            title = f'{event["step"]:02d} — {event["stage"]} — {event["elapsed_seconds"]:.3f}s'
            body = json.dumps(event, ensure_ascii=False, indent=2)
            display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>" + escape(body) + "</pre></details>"))
            displayed += 1
    print("Selected query:", selected_id, "Steps:", displayed)
    print("Available IDs:", ", ".join(sample_ids))
    print("Full log:", step_file)
